# Camera Pose Estimation with Pose Transformer

This notebook demonstrates training the Pose Transformer for **camera pose estimation**
(also known as visual localization or camera relocalization).

Given an RGB image, the model predicts the camera's 6DoF pose:
- **Rotation**: Camera orientation in world coordinates (SO(3))
- **Translation**: Camera position in world coordinates (R(3))

## Applications
- Visual SLAM and localization
- Autonomous navigation
- Augmented reality
- Structure from Motion (SfM)

## Standard Datasets
- **7-Scenes**: Indoor RGB-D dataset with 7 different scenes
- **Cambridge Landmarks**: Outdoor urban scenes
- **12-Scenes**: Indoor dataset with ground truth from RGB-D SLAM

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation
from tqdm.auto import tqdm
import random
import os

# Set seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Synthetic Camera Pose Dataset

We create a synthetic dataset simulating a camera moving through a 3D environment.
The camera observes a scene with feature points, and we render synthetic views.

In practice, you would use real datasets like 7-Scenes or Cambridge Landmarks.

In [ ]:
class SyntheticCameraPoseDataset(Dataset):
    """
    Synthetic dataset for camera pose estimation.
    
    Simulates a camera moving through a 3D scene with known poses.
    The scene contains randomly distributed 3D points that are projected
    into the camera view.
    """
    
    def __init__(self, num_samples=5000, image_size=224, 
                 num_scene_points=200, scene_size=10.0):
        self.num_samples = num_samples
        self.image_size = image_size
        self.scene_size = scene_size
        
        # Generate random 3D scene points (simulating landmarks)
        self.scene_points = np.random.uniform(
            -scene_size/2, scene_size/2, (num_scene_points, 3)
        ).astype(np.float32)
        
        # Assign random colors to scene points
        self.point_colors = np.random.rand(num_scene_points, 3).astype(np.float32)
        
        # Generate camera trajectory (smooth path through scene)
        self.camera_rotations = []
        self.camera_translations = []
        
        self._generate_camera_trajectory()
        
        # Camera intrinsics
        self.fx = self.fy = image_size * 1.5
        self.cx = self.cy = image_size / 2
        self.K = np.array([
            [self.fx, 0, self.cx],
            [0, self.fy, self.cy],
            [0, 0, 1]
        ], dtype=np.float32)
    
    def _generate_camera_trajectory(self):
        """Generate a smooth camera trajectory through the scene."""
        # Generate trajectory as interpolated waypoints
        num_waypoints = 10
        t = np.linspace(0, 2 * np.pi, num_waypoints)
        
        # Spiral trajectory
        waypoint_x = 3 * np.cos(t)
        waypoint_y = 3 * np.sin(t)
        waypoint_z = np.linspace(-2, 2, num_waypoints)
        waypoints = np.stack([waypoint_x, waypoint_y, waypoint_z], axis=1)
        
        for i in range(self.num_samples):
            # Random position on trajectory with noise
            t = np.random.uniform(0, 2 * np.pi)
            x = 3 * np.cos(t) + np.random.normal(0, 0.3)
            y = 3 * np.sin(t) + np.random.normal(0, 0.3)
            z = np.random.uniform(-2, 2)
            translation = np.array([x, y, z], dtype=np.float32)
            
            # Camera looks roughly toward center with some variation
            look_at = np.array([0, 0, 0]) + np.random.normal(0, 0.5, 3)
            forward = look_at - translation
            forward = forward / (np.linalg.norm(forward) + 1e-8)
            
            # Construct rotation matrix (camera to world)
            up = np.array([0, 0, 1], dtype=np.float32)
            right = np.cross(forward, up)
            right = right / (np.linalg.norm(right) + 1e-8)
            up = np.cross(right, forward)
            
            rotation = np.stack([right, up, -forward], axis=1).astype(np.float32)
            
            self.camera_rotations.append(rotation)
            self.camera_translations.append(translation)
        
        self.camera_rotations = np.array(self.camera_rotations)
        self.camera_translations = np.array(self.camera_translations)
    
    def __len__(self):
        return self.num_samples
    
    def _render_view(self, rotation, translation):
        """Render the scene from a given camera pose."""
        image = np.zeros((self.image_size, self.image_size, 3), dtype=np.float32)
        
        # Transform scene points to camera coordinates
        # World to camera: p_cam = R^T @ (p_world - t)
        points_world = self.scene_points
        points_cam = (rotation.T @ (points_world - translation).T).T
        
        # Filter points in front of camera
        valid_mask = points_cam[:, 2] > 0.1
        points_cam = points_cam[valid_mask]
        colors = self.point_colors[valid_mask]
        
        # Project to image
        points_2d = (self.K @ points_cam.T).T
        points_2d = points_2d[:, :2] / points_2d[:, 2:3]
        
        # Render points as circles
        for p2d, depth, color in zip(points_2d, points_cam[:, 2], colors):
            x, y = int(p2d[0]), int(p2d[1])
            if 0 <= x < self.image_size and 0 <= y < self.image_size:
                # Point size inversely proportional to depth
                radius = max(1, int(10 / depth))
                self._draw_circle(image, x, y, radius, color)
        
        # Add some noise
        noise = np.random.randn(*image.shape).astype(np.float32) * 0.02
        image = np.clip(image + noise, 0, 1)
        
        return image
    
    def _draw_circle(self, image, cx, cy, radius, color):
        """Draw a filled circle with soft edges."""
        for y in range(max(0, cy - radius - 1), min(self.image_size, cy + radius + 2)):
            for x in range(max(0, cx - radius - 1), min(self.image_size, cx + radius + 2)):
                dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
                if dist <= radius:
                    alpha = 1.0 - (dist / (radius + 1))
                    image[y, x] = alpha * np.array(color) + (1 - alpha) * image[y, x]
    
    def __getitem__(self, idx):
        rotation = self.camera_rotations[idx]
        translation = self.camera_translations[idx]
        
        # Render image
        image = self._render_view(rotation, translation)
        
        # Convert to tensors
        image = torch.from_numpy(image).permute(2, 0, 1)
        rotation = torch.from_numpy(rotation)
        translation = torch.from_numpy(translation)
        
        return image, rotation, translation


# Create datasets
train_dataset = SyntheticCameraPoseDataset(num_samples=4000)
val_dataset = SyntheticCameraPoseDataset(num_samples=1000)

print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Scene points: {len(train_dataset.scene_points)}')

In [ ]:
# Visualize the scene and camera trajectory
fig = plt.figure(figsize=(16, 6))

# 3D scene visualization
ax1 = fig.add_subplot(121, projection='3d')

# Plot scene points
ax1.scatter(
    train_dataset.scene_points[:, 0],
    train_dataset.scene_points[:, 1],
    train_dataset.scene_points[:, 2],
    c=train_dataset.point_colors,
    s=10, alpha=0.5
)

# Plot camera positions (subset)
indices = np.random.choice(len(train_dataset), 100, replace=False)
cam_positions = train_dataset.camera_translations[indices]
ax1.scatter(
    cam_positions[:, 0],
    cam_positions[:, 1],
    cam_positions[:, 2],
    c='red', s=20, marker='^', label='Cameras'
)

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Scene with Camera Positions')
ax1.legend()

# Sample rendered views
ax2 = fig.add_subplot(122)
grid_size = 3
combined = np.zeros((train_dataset.image_size * grid_size, 
                     train_dataset.image_size * grid_size, 3))

for i in range(grid_size):
    for j in range(grid_size):
        idx = i * grid_size + j
        image, _, _ = train_dataset[idx * 100]
        y_start = i * train_dataset.image_size
        x_start = j * train_dataset.image_size
        combined[y_start:y_start + train_dataset.image_size,
                 x_start:x_start + train_dataset.image_size] = image.permute(1, 2, 0).numpy()

ax2.imshow(combined)
ax2.set_title('Sample Rendered Views')
ax2.axis('off')

plt.tight_layout()
plt.show()

## 2. Pose Transformer Model

We use the same Pose Transformer architecture, but with different scale
for translation since camera positions can be larger than object translations.

In [ ]:
from examples.pose_transformer import (
    PoseTransformer,
    pose_loss,
    rotation_6d_to_matrix,
    geodesic_loss_rotmat,
)

# Create model
model = PoseTransformer(
    image_size=224,
    patch_size=16,
    in_channels=3,
    embed_dim=256,
    depth=8,
    num_heads=4,
    rotation_repr='6d',
    num_objects=1,
    translation_scale=5.0,  # Larger scale for camera positions
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## 3. Training Setup

For camera pose estimation, we often use:
- **Learned weighting** between rotation and translation losses (PoseNet approach)
- **Separate learning rates** for backbone and pose heads

In [ ]:
# Hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 5e-5
NUM_EPOCHS = 30
WEIGHT_DECAY = 1e-4

# Learned loss weights (PoseNet-style)
# log(s) is learned, actual weight is exp(-s)
class LearnedLossWeights(nn.Module):
    """Learned loss weights as in PoseNet."""
    def __init__(self, init_rot=0.0, init_trans=0.0):
        super().__init__()
        self.s_rot = nn.Parameter(torch.tensor(init_rot))
        self.s_trans = nn.Parameter(torch.tensor(init_trans))
    
    def forward(self, rot_loss, trans_loss):
        # Loss = exp(-s) * L + s (regularization)
        weighted_rot = torch.exp(-self.s_rot) * rot_loss + self.s_rot
        weighted_trans = torch.exp(-self.s_trans) * trans_loss + self.s_trans
        return weighted_rot + weighted_trans

loss_weights = LearnedLossWeights().to(device)

# Data loaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

# Optimizer includes loss weights
optimizer = optim.AdamW(
    list(model.parameters()) + list(loss_weights.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
def train_epoch(model, loss_weights, loader, optimizer, device):
    """Train for one epoch with learned loss weights."""
    model.train()
    loss_weights.train()
    
    total_loss = 0
    total_rot_loss = 0
    total_trans_loss = 0
    num_samples = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, gt_rotation, gt_translation in pbar:
        images = images.to(device)
        gt_rotation = gt_rotation.to(device)
        gt_translation = gt_translation.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        pred_rotation, pred_translation = model(images)
        
        # Compute individual losses
        pred_mat = rotation_6d_to_matrix(pred_rotation)
        rot_loss = geodesic_loss_rotmat(pred_mat, gt_rotation)
        trans_loss = nn.functional.mse_loss(pred_translation, gt_translation)
        
        # Apply learned weights
        loss = loss_weights(rot_loss, trans_loss)
        
        loss.backward()
        optimizer.step()
        
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_rot_loss += rot_loss.item() * batch_size
        total_trans_loss += trans_loss.item() * batch_size
        num_samples += batch_size
        
        pbar.set_postfix({
            'loss': loss.item(),
            's_r': loss_weights.s_rot.item(),
            's_t': loss_weights.s_trans.item()
        })
    
    return {
        'loss': total_loss / num_samples,
        'rot_loss': total_rot_loss / num_samples,
        'trans_loss': total_trans_loss / num_samples,
    }


def evaluate(model, loader, device):
    """Evaluate the model."""
    model.eval()
    
    total_rot_error = 0  # Angular error in degrees
    total_trans_error = 0  # Euclidean distance in meters
    num_samples = 0
    
    all_rot_errors = []
    all_trans_errors = []
    
    with torch.no_grad():
        for images, gt_rotation, gt_translation in tqdm(loader, desc='Evaluating'):
            images = images.to(device)
            gt_rotation = gt_rotation.to(device)
            gt_translation = gt_translation.to(device)
            
            # Forward pass
            pred_rotation, pred_translation = model(images)
            pred_mat = rotation_6d_to_matrix(pred_rotation)
            
            # Angular error
            diff = torch.bmm(pred_mat.transpose(-2, -1), gt_rotation)
            trace = diff[:, 0, 0] + diff[:, 1, 1] + diff[:, 2, 2]
            cos_angle = (trace - 1.0) / 2.0
            cos_angle = torch.clamp(cos_angle, -1.0, 1.0)
            angle_error = torch.acos(cos_angle) * 180.0 / np.pi
            
            # Translation error
            trans_error = torch.norm(pred_translation - gt_translation, dim=-1)
            
            all_rot_errors.extend(angle_error.cpu().numpy())
            all_trans_errors.extend(trans_error.cpu().numpy())
            
            batch_size = images.size(0)
            total_rot_error += angle_error.sum().item()
            total_trans_error += trans_error.sum().item()
            num_samples += batch_size
    
    return {
        'rot_error_deg': total_rot_error / num_samples,
        'trans_error': total_trans_error / num_samples,
        'median_rot_error': np.median(all_rot_errors),
        'median_trans_error': np.median(all_trans_errors),
    }

## 4. Training Loop

In [ ]:
# Training history
history = {
    'train_loss': [], 'train_rot_loss': [], 'train_trans_loss': [],
    'val_rot_error': [], 'val_trans_error': [],
    'median_rot_error': [], 'median_trans_error': [],
    's_rot': [], 's_trans': []
}

best_combined_error = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch + 1}/{NUM_EPOCHS}')
    print('-' * 60)
    
    # Train
    train_metrics = train_epoch(model, loss_weights, train_loader, optimizer, device)
    
    # Evaluate
    val_metrics = evaluate(model, val_loader, device)
    
    # Update scheduler
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_metrics['loss'])
    history['train_rot_loss'].append(train_metrics['rot_loss'])
    history['train_trans_loss'].append(train_metrics['trans_loss'])
    history['val_rot_error'].append(val_metrics['rot_error_deg'])
    history['val_trans_error'].append(val_metrics['trans_error'])
    history['median_rot_error'].append(val_metrics['median_rot_error'])
    history['median_trans_error'].append(val_metrics['median_trans_error'])
    history['s_rot'].append(loss_weights.s_rot.item())
    history['s_trans'].append(loss_weights.s_trans.item())
    
    print(f"Train - Loss: {train_metrics['loss']:.4f}")
    print(f"Val   - Rotation: {val_metrics['rot_error_deg']:.2f}° (median: {val_metrics['median_rot_error']:.2f}°)")
    print(f"Val   - Translation: {val_metrics['trans_error']:.3f}m (median: {val_metrics['median_trans_error']:.3f}m)")
    print(f"Learned weights - s_rot: {loss_weights.s_rot.item():.3f}, s_trans: {loss_weights.s_trans.item():.3f}")
    
    # Save best model (using combined metric)
    combined_error = val_metrics['median_rot_error'] + val_metrics['median_trans_error'] * 10
    if combined_error < best_combined_error:
        best_combined_error = combined_error
        torch.save({
            'model': model.state_dict(),
            'loss_weights': loss_weights.state_dict(),
        }, 'camera_pose_transformer_best.pth')
        print(f'Saved best model!')

print(f'\nTraining complete!')

## 5. Visualize Training

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Rotation error
axes[0, 0].plot(history['val_rot_error'], label='Mean')
axes[0, 0].plot(history['median_rot_error'], label='Median')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Rotation Error (degrees)')
axes[0, 0].set_title('Validation Rotation Error')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Translation error
axes[0, 1].plot(history['val_trans_error'], label='Mean')
axes[0, 1].plot(history['median_trans_error'], label='Median')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Translation Error (meters)')
axes[0, 1].set_title('Validation Translation Error')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Training losses
axes[1, 0].plot(history['train_rot_loss'], label='Rotation')
axes[1, 0].plot(history['train_trans_loss'], label='Translation')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Training Losses')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Learned weights
axes[1, 1].plot(history['s_rot'], label='s_rot')
axes[1, 1].plot(history['s_trans'], label='s_trans')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Learned Weight (s)')
axes[1, 1].set_title('Learned Loss Weights')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## 6. Visualize Camera Trajectory Predictions

In [ ]:
def visualize_trajectory(model, dataset, device, num_samples=100):
    """Visualize predicted vs ground truth camera trajectory."""
    model.eval()
    
    gt_positions = []
    pred_positions = []
    
    indices = np.linspace(0, len(dataset) - 1, num_samples, dtype=int)
    
    for idx in tqdm(indices, desc='Predicting'):
        image, gt_rotation, gt_translation = dataset[idx]
        image_tensor = image.unsqueeze(0).to(device)
        
        with torch.no_grad():
            _, pred_translation = model(image_tensor)
        
        gt_positions.append(gt_translation.numpy())
        pred_positions.append(pred_translation[0].cpu().numpy())
    
    gt_positions = np.array(gt_positions)
    pred_positions = np.array(pred_positions)
    
    # 3D visualization
    fig = plt.figure(figsize=(14, 6))
    
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.scatter(gt_positions[:, 0], gt_positions[:, 1], gt_positions[:, 2],
                c='green', s=20, label='Ground Truth', alpha=0.7)
    ax1.scatter(pred_positions[:, 0], pred_positions[:, 1], pred_positions[:, 2],
                c='red', s=20, label='Predicted', alpha=0.7)
    
    # Draw error lines
    for gt, pred in zip(gt_positions[::5], pred_positions[::5]):
        ax1.plot([gt[0], pred[0]], [gt[1], pred[1]], [gt[2], pred[2]], 
                 'k-', alpha=0.2, linewidth=0.5)
    
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    ax1.set_title('Camera Trajectory: GT vs Predicted')
    ax1.legend()
    
    # 2D top-down view
    ax2 = fig.add_subplot(122)
    ax2.scatter(gt_positions[:, 0], gt_positions[:, 1], c='green', s=20, 
                label='Ground Truth', alpha=0.7)
    ax2.scatter(pred_positions[:, 0], pred_positions[:, 1], c='red', s=20,
                label='Predicted', alpha=0.7)
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_title('Top-Down View (XY plane)')
    ax2.legend()
    ax2.axis('equal')
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Compute statistics
    errors = np.linalg.norm(gt_positions - pred_positions, axis=1)
    print(f"\nTranslation Error Statistics:")
    print(f"  Mean:   {np.mean(errors):.3f} m")
    print(f"  Median: {np.median(errors):.3f} m")
    print(f"  Std:    {np.std(errors):.3f} m")
    print(f"  Max:    {np.max(errors):.3f} m")


# Load best model and visualize
checkpoint = torch.load('camera_pose_transformer_best.pth')
model.load_state_dict(checkpoint['model'])
visualize_trajectory(model, val_dataset, device)

## 7. Comparison with Different Datasets

For real-world evaluation, you would test on standard benchmarks:

### 7-Scenes Dataset
```python
# Expected results on 7-Scenes (PoseNet paper baseline):
# Scene      | Rotation Error | Translation Error
# Chess      | 4.48°          | 0.13m
# Fire       | 11.3°          | 0.27m
# Heads      | 13.0°          | 0.17m
# Office     | 5.55°          | 0.19m
# Pumpkin    | 4.75°          | 0.26m
# Kitchen    | 5.35°          | 0.27m
# Stairs     | 12.4°          | 0.35m
```

### Cambridge Landmarks
```python
# Expected results (outdoor scenes are harder):
# Scene          | Rotation Error | Translation Error
# King's College | 1.04°          | 0.88m
# Old Hospital   | 1.51°          | 2.57m
# Shop Facade    | 1.13°          | 1.18m
# St Mary's      | 1.63°          | 2.11m
```

In [ ]:
# Template for loading 7-Scenes dataset
class SevenScenesDataset(Dataset):
    """
    7-Scenes dataset loader (template).
    
    Download from: https://www.microsoft.com/en-us/research/project/rgb-d-dataset-7-scenes/
    
    Directory structure:
    7scenes/
        chess/
            seq-01/
                frame-000000.color.png
                frame-000000.pose.txt
                ...
    """
    
    def __init__(self, root_dir, scene, split='train', transform=None):
        self.root_dir = root_dir
        self.scene = scene
        self.split = split
        self.transform = transform
        
        # Load frame list based on split
        # Training: sequences 1-4
        # Testing: sequences 5+
        self.frames = self._load_frame_list()
    
    def _load_frame_list(self):
        """Load list of frames for this split."""
        frames = []
        # Implementation depends on dataset structure
        return frames
    
    def _load_pose(self, pose_file):
        """Load 4x4 pose matrix from file."""
        pose = np.loadtxt(pose_file).astype(np.float32)
        rotation = pose[:3, :3]
        translation = pose[:3, 3]
        return rotation, translation
    
    def __len__(self):
        return len(self.frames)
    
    def __getitem__(self, idx):
        frame_info = self.frames[idx]
        
        # Load image
        # image = Image.open(frame_info['color_path'])
        
        # Load pose
        # rotation, translation = self._load_pose(frame_info['pose_path'])
        
        # Apply transforms
        # if self.transform:
        #     image = self.transform(image)
        
        # return image, rotation, translation
        pass


print("7-Scenes dataset template defined.")
print("To use: download the dataset and implement _load_frame_list()")

## Summary

This notebook demonstrated:

1. **Camera Pose Estimation**: Predicting where a camera is in 3D space from an image
2. **Synthetic Scene Generation**: Creating training data with known ground truth
3. **Learned Loss Weighting**: Automatically balancing rotation and translation losses
4. **Trajectory Visualization**: Comparing predicted vs ground truth camera paths

### Key Differences from Object Pose:
- **Scale**: Camera translations can be several meters, not centimeters
- **Scene Dependence**: Model learns scene-specific features
- **Evaluation**: Median errors are often more meaningful than mean

### Improvements for Production:
- Use pretrained backbone (e.g., EfficientNet, ResNet)
- Multi-scale feature extraction
- Scene coordinate regression (DSAC++, etc.)
- Test-time refinement with PnP + RANSAC
- Uncertainty estimation for robust localization